In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/AI_Project/DLA')

In [ ]:
!pip install rapidfuzz

# Utils

In [ ]:
import matplotlib.pyplot as plt

def plot_dataset(images, labels, grid_width, grid_height, figure_width, figure_height, y_hats=None, vocab=None, wspace=0.5, hspace=1):
  # images, labels는 torch.tensor 타입으로 들어온다고 가정
  f, ax = plt.subplots(grid_height, grid_width, squeeze=False)
  f.set_size_inches(figure_width, figure_height)
  img_idx = 0
  eos_idx = 2

  for i in range(0, grid_height):
    for j in range(0, grid_width):
      image = images[img_idx].cpu().detach()
      label = labels[img_idx].cpu().detach()
      title_color = 'k'

      assert vocab is not None, "vocab을 입력하세요"

      label = label.tolist()
      label = label[1:label.index(eos_idx)]     # <sos>, <eos> 부분 제외 (EOS_IDX == 2) - <eos> 이전까지 뽑아서 <eos>이후 <pad> 부분도 제외되도록
      label = vocab.decode(label)               # list -> str

      if y_hats is not None:
            y_hat = y_hats[img_idx].cpu().detach().tolist()
            if eos_idx in y_hat:
                y_hat = y_hat[:y_hat.index(eos_idx)]   # <eos> 존재하면 <eos> 이후부분 (<eos> 및 <pad>) 무시

            y_hat = vocab.decode(y_hat)         # list -> str

            if label != y_hat:
                title_color = 'r'

            label = f'gt: {label}\npred:{y_hat}'

      ax[i][j].axis('off')
      ax[i][j].set_title(label, color=title_color)
      ax[i][j].imshow(image.permute(1,2,0).numpy(), aspect='auto', cmap='gray')
      img_idx += 1

    plt.subplots_adjust(left=0, bottom=0, right=2, top=2, wspace=wspace, hspace=hspace)
  plt.show()

In [ ]:
import torch
import numpy as np
import editdistance
from rapidfuzz.distance import Levenshtein as RLev

def _to_list_batch(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x

def strip_special(seq, pad_id=0, sos_id=1, eos_id=2):
    """
    PAD 제거, 맨 앞 SOS 제거, 첫 EOS 이전까지만 사용
    """
    out = [t for t in seq if t != pad_id]
    if sos_id is not None and len(out) and out[0] == sos_id:
        out = out[1:]
    if eos_id is not None and eos_id in out:
        out = out[:out.index(eos_id)]
    return out

# def exprate_k(preds, targets, k=0, pad_id=0, sos_id=1, eos_id=2):
#     """
#     Expression rate-k: 예측 수식과 정답 수식이 k개 이하의 토큰 차이만 있을 때 정답으로 간주
#     """
#     preds = _to_list_batch(preds)
#     targets = _to_list_batch(targets)
#     assert len(preds) == len(targets)
#     N = len(preds)
#     cnt = 0
#     for p, t in zip(preds, targets):
#         pred = strip_special(p, pad_id, sos_id, eos_id)
#         target = strip_special(t, pad_id, sos_id, eos_id)
#         dist = editdistance.eval(pred, target)
#         if dist <= k:
#             cnt += 1
#     return (cnt / N) if N > 0 else 0.0

# 왜 preds, targets에 <sos>, <eos>가 없다고 가정하는지?
def exprate_k(preds, targets, k=0, eos_idx=2):
    # preds, targets에는 <sos>가 제거된 채로 함수인자로 들어온다고 가정
    # preds: (B, T-1)
    # targets: (B, T-1)
    assert preds.shape[0] == targets.shape[0], "예측 label과 정답 label의 배치 크기가 다릅니다"
    cnt = 0
    for pred_seq, tgt_seq in zip(preds, targets):
        pred = pred_seq.tolist()
        tgt = tgt_seq.tolist()

        if eos_idx in pred:
            pred = pred[:pred.index(eos_idx)]

        if eos_idx in tgt:
            tgt = tgt[:tgt.index(eos_idx)]

        diff = editdistance.eval(pred, tgt)

        if diff<=k:
            cnt += 1
    return cnt / preds.shape[0]

def wer(preds, targets, pad_id=0, sos_id=None, eos_id=None):
    """
    WER = (N_sub + N_del + N_ins) / N_Y
    (토큰 단위. 특수토큰 제외한 정답 토큰 수 N_Y 기준)
    """
    preds = _to_list_batch(preds)
    targets = _to_list_batch(targets)

    D_sum, N_Y = 0, 0
    for p, t in zip(preds, targets):
        ref = strip_special(t, pad_id, sos_id, eos_id)  # target
        hyp = strip_special(p, pad_id, sos_id, eos_id)  # prediction
        D_sum += RLev.distance(ref, hyp)
        N_Y   += len(ref)
    return (D_sum / N_Y) if N_Y > 0 else 0.0

In [ ]:
import os
import json

def save_log(log_dict, save_path="log.json"):
    """학습 로그를 JSON으로 저장."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(log_dict, f, indent=2)
    print(f"로그 저장 완료: {save_path}")


def plot_loss_curve(log_dict, metric="loss", overlap=False):
    """Loss 곡선 시각화."""
    assert metric in ["loss", "expr@0", "expr@1", "expr@2", "expr@3"], "metric은 loss, expr@0, expr@2, expr@3 중 선택하여 입력해야합니다"

    if overlap==False:
        fig, axes = plt.subplots(2,1)
        fig.suptitle(f"{metric.upper()} Curve")

        # train
        train = log_dict["Train"]
        epochs = range(1, len(train)+1)
        train_metric = []
        for log in train:
            train_metric.append(log[metric])

        axes[0].plot(epochs, train_metric, marker='o')
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel(f"train {metric}")
        axes[0].grid(True)

        # valid
        valid = log_dict["Valid"]
        epochs = range(1, len(valid)+1)
        valid_metric = []
        for log in valid:
            valid_metric.append(log[metric])

        axes[1].plot(epochs, valid_metric, marker='o')
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel(f"valid {metric}")
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

    else:
        # train
        train = log_dict["Train"]
        epochs = range(1, len(train)+1)
        train_metric = []
        for log in train:
            train_metric.append(log[metric])

        plt.plot(epochs, train_metric, marker='o', label=f"Train {metric}")

        # valid
        valid = log_dict["Valid"]
        epochs = range(1, len(valid)+1)
        valid_metric = []
        for log in valid:
            valid_metric.append(log[metric])

        plt.plot(epochs, valid_metric, marker='o', label=f"Valid {metric}")

        plt.xlabel("Epoch")
        plt.ylabel(f"{metric}")
        plt.title(f"{metric.upper()} Curve")
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
import gspread

def save_test_result(experiment_name, test_dataset_name, val, metric="loss"):
    """ 실험결과를 구글스프레드시트에 저장 """
    assert test_dataset_name in ["CROHME_2014", "CROHME_2016", "CROHME_2019", "CROPME_2014", "CROPME_2016", "CROPME_2019", "IM2LATEX*"], \
        "테스트데이터셋 이름은 CROHME_2014, CROHME_2016, CROHME_2019, CROPME_2014, CROPME_2016, CROPME_2019, IM2LATEX* 중 하나여야 합니다"
    assert metric in ["loss", "expr@0", "expr@1", "expr@2", "expr@3"], "metric은 loss, expr@0, expr@2, expr@3 중 선택하여 입력해야합니다"

    gc = gspread.service_account(filename="service_account.json")
    sh = gc.open("test_results")
    worksheet = sh.worksheet(metric)

    # experiment_name이 스프레드시트에 존재하는지 확인
    try:
        row = worksheet.find(experiment_name).row
        # 이미 해당 실험이 과거에 진행된 경우가 있어 값들이 채워져있다면, 새로 실험하는 것이므로 해당 실험의 이전값들을 모든 sheet에 대해서 초기화해주고 시작
        cols = [worksheet.find(name).col for name in ["CROHME_2014", "CROHME_2016", "CROHME_2019", "CROPME_2014", "CROPME_2016", "CROPME_2019", "IM2LATEX*"]]

        for sheet_name in ["loss", "expr@0", "expr@1", "expr@2", "expr@3"]:
            ws = sh.worksheet(sheet_name)
            cell_range = f"{gspread.utils.rowcol_to_a1(row, min(cols))}:{gspread.utils.rowcol_to_a1(row, max(cols))}"
            ws.update(values=[[""] * len(cols)], range_name=cell_range)

    except:
        last_row = len(worksheet.col_values(1))  # A열 중 값이 채워진 부분의 마지막 행번호
        worksheet.update_cell(last_row+1, 1, experiment_name)
        row = last_row + 1

    col = worksheet.find(test_dataset_name).col
    worksheet.update_cell(row, col, val)

In [ ]:
gc = gspread.service_account(filename="service_account.json")
sh = gc.open("test_results")
worksheet = sh.worksheet("loss")

row = worksheet.find('tmp').row

cols = [worksheet.find(name).col for name in ["CROHME_2014", "CROHME_2016", "CROHME_2019", "CROPME_2014", "CROPME_2016", "CROPME_2019", "IM2LATEX*"]]

for sheet_name in ["loss", "expr@0", "expr@1", "expr@2", "expr@3"]:
    ws = sh.worksheet(sheet_name)
    cell_range = f"{gspread.utils.rowcol_to_a1(row, min(cols))}:{gspread.utils.rowcol_to_a1(row, max(cols))}"
    ws.update(values=[[""] * len(cols)], range_name=cell_range)

# Config (Global Parameters)

In [ ]:
import os, json
from datetime import datetime
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

CFG = {
  "experiment_name": "dla_baseline",
  "img_size": (128,768),
  "pad_idx": 0,
  "sos_idx": 1,
  "eos_idx": 2,

  "data": {
    "vocab": "data/vocab.txt",
    "paired": {
      "hme_img": "data/train/crohme/hme",
      "pme_img": "data/train/crohme/pme",
      "caption": "data/train/crohme/caption.txt",
    },
    "unpaired": {
      "pme_img": "data/train/im2latex/pme",
      "caption": "data/train/im2latex/caption.txt",
    },
  },

  "model": {
    "vocab_size": 113,
    "encoder_emb_dim": 256,
    "decoder_emb_dim": 256,
    "decoder_hidden_dim": 512
  },

  "training": {
    "batch_size": 8,  # 64
    "epochs": 10,
    "learning_rate": 0.001,
    "match_weight": 0.2,
    "optimizer": "adadelta",
    "grad_clip": 1.0,
    "tf_ratio": 0.5,
    "early_stop_patience": 4
    # "ignore_idx": 0,  # 필요?
    # "scheduler": {"use": True, "type": "StepLR", "step_size": 10, "gamma": 0.5} # 필요?
  },

  "testing": {"batch_size": 1, "max_len": 150},

  "paths":{
      "experiment_name": "dla_single_epoch100_bs16_scheduler_CosineAnnealingWarmupRestarts_patience_4",
      "best_ckpt": "runs/dla_single_epoch100_bs16_scheduler_CosineAnnealingWarmupRestarts_patience_4/best_model.pth",
      "last_ckpt": "runs/dla_single_epoch100_bs16_scheduler_CosineAnnealingWarmupRestarts_patience_4/last_model.pth",
      "train_log_json": "runs/dla_single_epoch100_bs16_scheduler_CosineAnnealingWarmupRestarts_patience_4/train_log.json"
  }
}

In [ ]:
# 실험환경 폴더 생성
from pathlib import Path

Path(os.path.join("runs", CFG["paths"]["experiment_name"])).mkdir(parents=True, exist_ok=True)

# Custom Dataset

## Vocab

In [ ]:
from typing import List
import re

# IM2LATEX 캡션 전처리용
def tokenize_formula(formula):
    ans = []
    tokens = formula.strip().split()

    for tok in tokens:
        chk = 0
        for sym in ["cm", "mm", "pt", "in", "ex", "em"]:
            if (tok[-2:] == sym) and tok[-3].isdigit():
                num, _ = tok.split(sym)
                num = ' '.join(num).split()
                ans.extend(num)
                ans.append(sym)
                chk = 1
                break

        if chk == 0:
            ans.append(tok)
    return ans

class Vocab:
  PAD_TOKEN = "<pad>"
  SOS_TOKEN = "<sos>"
  EOS_TOKEN = "<eos>"

  def __init__(self):
    self.default_tokens = [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN]
    self.token2idx = {tok: idx for idx, tok in enumerate(self.default_tokens)}
    self.idx2token = self.default_tokens.copy()

  def __len__(self):
    return len(self.idx2token)

  def load_from_txt(self, path):
    with open(path, "r", encoding="utf-8") as f:
      for line in f:
        token = line.strip()
        if token not in self.token2idx:
          idx = len(self.token2idx)
          self.token2idx[token] = idx
          self.idx2token.append(token)

  def encode(self, caption: str) -> List[int]:
    tokens = tokenize_formula(caption)
    return [self.token2idx[self.SOS_TOKEN]] + \
           [self.token2idx[token] for token in tokens] + \
           [self.token2idx[self.EOS_TOKEN]]

  def decode(self, token_ids: List[int]) -> str:
    return ' '.join([self.idx2token[token_id] for token_id in token_ids])

vocab = Vocab()
vocab.load_from_txt("data/vocab.txt")

In [ ]:
# from pathlib import Path
# from typing import List, Tuple

# # img_id, formula pair 반환
# def load_caption_pairs(caption_path: Path) -> List[Tuple[str, str]]:
#     pairs = []
#     with open(caption_path, "r", encoding="utf-8") as f:
#         for i, line in enumerate(f, 1):
#             s = line.rstrip("\n")
#             if not s:
#                 continue
#             parts = re.split(r"[\t,]", s, maxsplit=1)
#             if len(parts) < 2:
#                 parts = s.split(None, 1)
#             if len(parts) < 2:
#                 continue
#             img_id, formula = parts[0].strip(), parts[1].strip()
#             pairs.append((Path(img_id).stem, formula))
#     return pairs  # caption만 확인 -> {k: v for k, v in load_caption_pairs(caption_path)}

In [ ]:
# from pathlib import Path
# from collections import Counter

# """
# OOV 커버리지 체크 (caption 6가지)
# train/crohme, im2latex
# test/crohme_2014, 2016, 2019, im2latex
# """
# VOCAB_TXT   = Path("data/vocab.txt")
# CAPTION_TXT = Path("data/train/crohme/caption.txt")

# # vocab 로드 (파일 첫 3줄이 <pad>,<sos>,<eos>라고 가정)
# def load_vocab_txt(path: Path):
#     with open(path, "r", encoding="utf-8") as f:
#         toks = [line.strip() for line in f if line.strip()]
#     token2idx = {t:i for i,t in enumerate(toks)}
#     return toks, token2idx

# id2tok, tok2id = load_vocab_txt(VOCAB_TXT)

# # 스페셜 토큰 검증
# assert id2tok[0] == "<pad>" and id2tok[1] == "<sos>" and id2tok[2] == "<eos>", \
#     f"Different order of special tokens: {id2tok[:3]}"
# print(f"[INFO] vocab size: {len(id2tok)} (no <unk>)")
# caps = load_caption_pairs(CAPTION_TXT)
# print(f"[INFO] captions: {len(caps)} lines")

# # OOV 스캔
# oov_counter = Counter()
# covered_counter = Counter()
# oov_by_sample = []  # (img_id, [oov tokens])

# total_tokens = 0
# for img_id, formula in caps:
#     toks = tokenize_formula(formula)
#     total_tokens += len(toks)
#     oov_toks = [t for t in toks if t not in tok2id]
#     if oov_toks:
#         oov_counter.update(oov_toks)
#         oov_by_sample.append((img_id, oov_toks))
#     else:
#         covered_counter.update(toks)

# unique_tokens = len(set(t for _, f in caps for t in tokenize_formula(f)))
# covered_unique = unique_tokens - len(oov_counter)
# covered_total = total_tokens - sum(oov_counter.values())

# # 리포트
# print(f"Unique tokens     : {unique_tokens}")
# print(f"Covered unique    : {covered_unique}  ({covered_unique/unique_tokens*100:.2f}%)")
# print(f"Total tokens      : {total_tokens}")
# print(f"Covered total     : {covered_total}  ({covered_total/total_tokens*100:.2f}%)")
# print(f"OOV unique count  : {len(oov_counter)}")
# print(f"OOV total count   : {sum(oov_counter.values())}")

# if oov_counter:
#     print("\nTop-30 OOV tokens (by frequency):")
#     for tok, cnt in oov_counter.most_common(30):
#         print(f"  {tok!r}: {cnt}")
# else:
#     print("\nNo OOV found.")

In [ ]:
import torch

def unpaired_collate_fn(batch):
    images = torch.stack([item["image"] for item in batch], dim=0)
    formulas = torch.nn.utils.rnn.pad_sequence(
        [item["formula"] for item in batch],
        batch_first=True,
        padding_value=CFG["pad_idx"]
    )
    return {"image": images, "formula": formulas}

def paired_collate_fn(batch):
    imgs_hme = torch.stack([item["img_hme"] for item in batch], dim=0)
    imgs_pme = torch.stack([item["img_pme"] for item in batch], dim=0)
    formulas = torch.nn.utils.rnn.pad_sequence(
        [item["formula"] for item in batch],
        batch_first=True,
        padding_value=CFG["pad_idx"]
    )
    return {"img_hme": imgs_hme, "img_pme": imgs_pme, "formula": formulas}

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import torch
import os
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(CFG["img_size"]),
    transforms.Normalize(mean=[0.5], std=[0.5]),]
)

class UnpairedDataset(Dataset):
  def __init__(self, img_path, labels_path, transform, vocab):
    self.img_path = img_path
    self.labels_path = labels_path
    self.transform = transform
    self.vocab = vocab

    self.samples = []
    with open(labels_path, "r", encoding="utf-8") as f:
      for line in f:
        parts = line.strip().split('\t')
        if len(parts)!=2:
          continue
        file_id, caption = parts

        if '.' not in file_id:
          file_id = f"{file_id}.png"

        img_path = os.path.join(self.img_path, file_id)
        if os.path.exists(img_path):
          self.samples.append((img_path, caption))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    img_path, caption = self.samples[idx]
    img = Image.open(img_path).convert('L')

    if self.transform:
      img = self.transform(img)

    caption_tokens = self.vocab.encode(caption)
    caption = torch.tensor(caption_tokens)

    return {"image": img, "formula": caption}

class PairedDataset(Dataset):
    def __init__(self, hme_path, pme_path, labels_path, transform, vocab=None):
        self.hme_path = Path(hme_path)
        self.pme_path = Path(pme_path)
        self.labels_path = labels_path
        self.transform = transform
        self.vocab = vocab

        self.samples = []
        with open(labels_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) != 2:
                    continue
                file_id, caption = parts

                if '.' not in file_id:
                    file_id = f"{file_id}.png"

                img_hme_path = os.path.join(self.hme_path, file_id)
                img_pme_path = os.path.join(self.pme_path, file_id)
                if os.path.exists(img_hme_path) and os.path.exists(img_pme_path):
                  self.samples.append((img_hme_path, img_pme_path, caption))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_hme_path, img_pme_path, caption = self.samples[idx]
        img_hme = Image.open(img_hme_path).convert('L')
        img_pme = Image.open(img_pme_path).convert('L')

        if self.transform:
            img_hme = self.transform(img_hme)
            img_pme = self.transform(img_pme)

        caption_tokens = self.vocab.encode(caption)
        caption = torch.tensor(caption_tokens)

        return {"img_hme": img_hme, "img_pme": img_pme, "formula": caption}

In [ ]:
train_hme_path = CFG["data"]["paired"]["hme_img"]
train_pme_path = CFG["data"]["paired"]["pme_img"]
train_labels_path = CFG["data"]["paired"]["caption"]

years = [2014, 2016, 2019]
crohme_test_paths = [f"data/test/crohme/hme/{year}" for year in years]
crohme_test_labels_paths = [f"data/test/crohme/caption_{year}.txt" for year in years]
cropme_test_paths = [f"data/test/crohme/pme/{year}" for year in years]
cropme_test_labels_paths = [f"data/test/crohme/caption_{year}.txt" for year in years]

batch_size = CFG["training"]["batch_size"]

# train/valid split
trainset = PairedDataset(train_hme_path, train_pme_path, train_labels_path, transform, vocab)
train_size = int(0.8 * len(trainset))
valid_size = len(trainset) - train_size
trainset, validset = torch.utils.data.random_split(trainset,[train_size, valid_size])

# train, valid
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, collate_fn=paired_collate_fn
)
validloader = torch.utils.data.DataLoader(
    validset, batch_size=1, shuffle=False, collate_fn=paired_collate_fn
)

# test - CROHME
crohme_testset_2014 = UnpairedDataset(crohme_test_paths[0], crohme_test_labels_paths[0], transform, vocab)
crohme_testset_2016 = UnpairedDataset(crohme_test_paths[1], crohme_test_labels_paths[1], transform, vocab)
crohme_testset_2019 = UnpairedDataset(crohme_test_paths[2], crohme_test_labels_paths[2], transform, vocab)

crohme_testloader_2014 = torch.utils.data.DataLoader(
    crohme_testset_2014, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn,
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

crohme_testloader_2016 = torch.utils.data.DataLoader(
    crohme_testset_2016, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn.
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

crohme_testloader_2019 = torch.utils.data.DataLoader(
    crohme_testset_2019, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn,
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

# test - CROPME
cropme_testset_2014 = UnpairedDataset(cropme_test_paths[0], cropme_test_labels_paths[0], transform, vocab)
cropme_testset_2016 = UnpairedDataset(cropme_test_paths[1], cropme_test_labels_paths[1], transform, vocab)
cropme_testset_2019 = UnpairedDataset(cropme_test_paths[2], cropme_test_labels_paths[2], transform, vocab)

cropme_testloader_2014 = torch.utils.data.DataLoader(
    cropme_testset_2014, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn,
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

cropme_testloader_2016 = torch.utils.data.DataLoader(
    cropme_testset_2016, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn,
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

cropme_testloader_2019 = torch.utils.data.DataLoader(
    cropme_testset_2019, batch_size=1, shuffle=False, collate_fn=unpaired_collate_fn,
    num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2
)

In [ ]:
# # pairedDataset 정상적으로 작동하는지 확인
# batch = next(iter(trainloader))
# images_hme = batch['img_hme']
# images_pme = batch['img_pme']
# labels = batch['formula']
# plot_dataset(images_hme, labels, grid_width=1, grid_height=8, figure_width=1, figure_height=8, vocab=vocab, wspace=0.25, hspace=0.25)
# plot_dataset(images_pme, labels, grid_width=1, grid_height=8, figure_width=1, figure_height=8, vocab=vocab, wspace=0.25, hspace=0.25)

# MODEL

## Encoder (사전학습 가중치 O)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet121, DenseNet121_Weights

class DenseNetEncoder(nn.Module):
    """
    Input : x -> (B, 1, H=128, W)
    Output : seq -> (B, C=1024, H'=4, W')
    """
    def __init__(self, pretrained: bool=True):
        super().__init__()

        # 사전학습 가중치 선택 (A/B 테스트 돌려보면 좋을 듯?)
        w = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        densenet = densenet121(weights=w)

        # conv0 가중치만 미리 저장 -> 1채널 conv 교체 후 복사
        old_w = densenet.features.conv0.weight.detach().clone()
        densenet.features.conv0 = nn.Conv2d(
            in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False
        )

        if pretrained:
            with torch.no_grad():
                # gpu 메모리 소모 줄이기 위해 그레이스케일 변환
                densenet.features.conv0.weight.copy_(old_w.mean(dim=1, keepdim=True))

        # Remove classification head
        features = list(densenet.features.children())

        # Use all layers except final norm+relu+avgpool
        self.backbone = nn.Sequential(*features[:-1])

        # 사전학습 가중치 사용안하면 초기화
        if not pretrained:
            self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    # 메모리 연산량이 더 적음
    def forward(self, x):
        assert x.dim() == 4, f"Encoder expects 4D (B,C,H,W), got {tuple(x.shape)}"
        x = self.backbone(x)  # (B, C, H', W')
        B, C, Hp, Wp = x.shape
        x = x.permute(0, 3, 2, 1).contiguous().view(B, Wp, Hp * C)  # (B, W', H'*C)
        # print("\nEncoder output size: ", x.shape)
        return x

## Decoder

In [ ]:
import torch
import torch.nn as nn

# 차원 숫자를 CFG랑 통일해야하지 않을까?
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim_enc=256, hidden_dim=512, emb_dim_dec=256, num_layers=1):
        super().__init__()
        self.emb_dim_enc = emb_dim_enc
        self.emb_dim_dec = emb_dim_dec
        self.hidden_dim  = hidden_dim
        self.num_layers  = num_layers

        self.gru = nn.GRU(emb_dim_enc + emb_dim_dec, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

        for name, param in self.gru.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, prev_token, hidden, context):
        """
        prev_token: (B, D_dec)  # 이미 임베딩된 벡터를 받는 전제 유지
        hidden    : (B, H) 또는 (L, B, H)
        context   : (B, D_enc) 또는 (B, 1, D_enc)
        """
        # prev_token: (B,D_dec)
        # hidden: (B,H)
        # context: (B,D_enc)

        # 지금은 num_layers는 무조건 1이라서 context는 (B,L,H) 대신 (B,1,H), hidden은 (D*L,B,H) 대신 (D,B,H)로 받는다고 생각하고 unsqueeze 등 shape 변경
        prev_token = prev_token.unsqueeze(1)                  # (B,D_dec) -> (B,1,D_dec)
        context = context.unsqueeze(1)                        # (B,D_enc) -> (B,1,D_enc)
        dec_input = torch.cat([prev_token, context], dim=-1)  # (B,1,D_dec+D_enc)

        # nn.GRU에서 요구하는 hidden 형상 (D*L,B,H)가 되도록 변경
        hidden = hidden.unsqueeze(0)
        output, hidden = self.gru(dec_input, hidden)
        output = output.squeeze(1)                       # (B,1,H) -> (B,H)
        hidden = hidden.squeeze(0)                       # (1,B,H) -> (B,H)
        output_logits = self.fc_out(output)             # (B,H) -> (B,V)
        return output_logits, hidden

## BahdanauAttention (1D)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    """
    enc : (B, T, D)  # 인코더 시퀀스
    prev_hidden : (B, H) 또는 (L, B, H)  # 디코더 히든(마지막 레이어 사용)
    mask   : (B, T) bool, True=유효
    returns:
      context : (B, D)
      attn    : (B, T)
    """
    def __init__(self, enc_emb_dim, dec_hid_dim):
        super().__init__()
        self.W_enc = nn.Linear(enc_emb_dim, dec_hid_dim, bias=False)
        self.W_hid = nn.Linear(dec_hid_dim, dec_hid_dim, bias=False)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

   @torch.no_grad()
    def precompute(self, enc: torch.Tensor) -> torch.Tensor:
        # enc: (B, T, D) -> (B, T, H)
        return self.W_enc(enc)

    def forward(self, enc, prev_hidden, mask=None, enc_proj=None):
        if prev_hidden.dim() == 3:        # (L,B,H)
            prev_hidden = prev_hidden[-1] # 마지막 레이어

        if enc_proj is None:
            enc_proj = self.W_enc(enc)    # (B, T, H)

        # scores: (B,T,H) -> (B,T)
        scores = torch.tanh(enc_proj + self.W_hid(prev_hidden).unsqueeze(1))
        scores = self.v(scores).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)                         # (B, T)
        context = torch.bmm(attn.unsqueeze(1), enc).squeeze(1)   # (B, 1, T) @ (B, T, D) -> (B, 1, D)
        return context, attn

## DLA

- 이후 추가 모델 개선방안 고민
  - contrastive learning 및 data augmentation (scale augmentation 등)
  - BYOL, DINO 같은 방법 참고해보기? (distilation)
  - DA, DG, SFTA 방법론 참고한 다른 시도
  - multi resolution 기반 실험

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

class DLAModel_Lite(nn.Module):
    def __init__(self, model_config=None):
        super().__init__()

        # 설정 파라미터 추출
        self.vocab_size = model_config.get("vocab_size", 113)
        self.enc_emb_dim = model_config.get("encoder_emb_dim", 256)
        self.dec_emb_dim = model_config.get("decoder_emb_dim", 256)
        self.dec_hid_dim = model_config.get("decoder_hidden_dim", 512)

        self.embed = nn.Embedding(self.vocab_size, self.dec_emb_dim)

        # 인코더
        self.encoder = DenseNetEncoder()
        self.encoder_proj = nn.Linear(4096, self.enc_emb_dim)

        # 바다나우 어텐션
        self.attn = BahdanauAttention(enc_dim=self.enc_emb_dim, dec_hidden_dim=self.dec_hid_dim)

        # 디코더
        self.decoder = Decoder(self.vocab_size, self.enc_emb_dim, self.dec_hid_dim, self.dec_emb_dim)

        # 초기화
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, images, labels, tf_ratio=CFG["training"]["tf_ratio"]):
        # 인코더
        enc = self.encoder(images)  # (B, T_enc, D_raw)
        enc = self.encoder_proj(enc)  # (B, T_enc, D_enc)

        # (선택) 마스크
        mask = None

        # 어텐션 미리 계산
        enc_proj = self.attn.precompute(enc)  # (B, T_enc, H)

        # 디코딩
        B = enc.size(0)
        T_lbl = labels.size(1)
        dec_out = []

        # <sos>
        prev_ids    = labels[:, 0]                          # (B,)
        prev_token  = self.embed(prev_ids)                  # (B,E)
        prev_hidden = torch.zeros((B, self.dec_hid_dim), device=images.device)  # (B,H)

        for t in range(1,T_lbl):
            # 어텐션 한 스텝
            context, attn_coeff = self.attn(enc, prev_hidden, mask=mask, enc_proj=enc_proj)  # (B,D_enc), (B,T_enc)

            # 디코더 한 스텝
            logits_y, cur_hidden = self.decoder(prev_token, prev_hidden, context)            # (B,V), (B,H)
            dec_out.append(logits_y)

            # teacher forcing 적용하는 경우
            if random.random() < tf_ratio:
                next_id = labels[:, t]             # (B, )
            else:
                next_id = logits_y.argmax(dim=-1)  # (B,V) -> (B,)
            prev_token = self.embed(next_id)       # (B,) -> (B,D_dec)
            prev_hidden = cur_hidden

        dec_out = torch.stack(dec_out).transpose(0,1)  # (T-1, B, V) -> (B, T-1, V)
        return dec_out

    def inference(self, images, max_seq_len):
        enc = self.encoder(images)     # (B, T, D_raw)
        enc = self.encoder_proj(enc)   # (B, T, D_enc)

        mask = None
        enc_proj = self.attn.precompute(enc)

        B = enc.size(0)
        dec_out = []

        sos_id = 1
        prev_ids    = torch.full((B,), sos_id, dtype=torch.long, device=images.device)
        prev_token  = self.embed(prev_ids)                                # (B,E)
        prev_hidden = torch.zeros((B, self.dec_hid_dim), device=images.device)

        # inference 시점이므로 teacher forcing 적용하지 않음
        for t in range(1,max_seq_len):
            context, attn_coeff = self.attn(enc, prev_hidden, mask=mask, enc_proj=enc_proj)
            logits_y, cur_hidden = self.decoder(prev_token, prev_hidden, context)
            dec_out.append(logits_y)

            next_ids   = logits_y.argmax(dim=-1)
            prev_token = self.embed(next_ids)
            prev_hidden = cur_hidden

        dec_out = torch.stack(dec_out).transpose(0,1)  # (B, T-1, V)
        return dec_out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DLAModel_Full(nn.Module):
    def __init__(self, model_config=None):
        super().__init__()

        # 설정 파라미터 추출
        self.vocab_size = model_config.get("vocab_size", 113) # V
        self.enc_emb_dim = model_config.get("encoder_emb_dim", 256) # C
        self.dec_emb_dim = model_config.get("decoder_emb_dim", 256)
        self.dec_hid_dim = model_config.get("decoder_hidden_dim", 512)

        # 임베딩
        self.embed = nn.Embedding(self.vocab_size, self.dec_emb_dim)

        # 인코더
        self.encoder = DenseNetEncoder()
        self.encoder_proj = nn.Linear(4096, self.enc_emb_dim)

        # 바다나우 어텐션
        self.attn = BahdanauAttention(enc_dim=self.enc_emb_dim, dec_hidden_dim=self.dec_hid_dim)

        # 디코더
        self.decoder = Decoder(self.vocab_size, self.enc_emb_dim, self.dec_hid_dim, self.dec_emb_dim)

        # 초기화
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _decode_loop(self, enc, labels, tf_ratio):
            """
            enc   : (B, T, D_enc)  # proj 이후
            labels: (B, T_lbl)     # <sos> 포함
            """
            B = enc.size(0)
            T_lbl = labels.size(1)
            dec_out = []

            prev_ids    = labels[:, 0]                          # (B,)
            prev_token  = self.embed(prev_ids)                  # (B,E)
            prev_hidden = torch.zeros((B, self.dec_hid_dim), device=images.device)  # (B,H)

            for t in range(1, T_lbl):
                # 어텐션 한 스텝
                context, attn_coeff = self.attn(enc, prev_hidden, mask=mask, enc_proj=enc_proj)  # (B,D_enc), (B,T_enc)

                # 디코더 한 스텝
                logits_y, cur_hidden = self.decoder(prev_token, prev_hidden, context)            # (B,V), (B,H)
                dec_out.append(logits_y)

                # teacher forcing
                if random.random() < tf_ratio:
                    next_id = labels[:, t]
                else:
                    next_id = logits_y.argmax(dim=-1)
                prev_token  = self.embed(next_id)
                prev_hidden = cur_hidden

            dec_out = torch.stack(dec_out).transpose(0, 1)  # (B, T_lbl-1, V)
            return dec_out

    def forward(self, images_h, labels_h, images_p, labels_p, tf_ratio=CFG["training"]["tf_ratio"]):
        """
        images_h: (B, 1, 128, W_h)  # HME
        labels_h: (B, T_h)
        images_p: (B, 1, 128, W_p)  # PME
        labels_p: (B, T_p)
        """
        # 인코더
        enc_h = self.encoder(images_h)         # (B, T_hfeat, D_raw)
        enc_h = self.encoder_proj(enc_h)       # (B, T_hfeat, D_enc)
        enc_p = self.encoder(images_p)         # (B, T_pfeat, D_raw)
        enc_p = self.encoder_proj(enc_p)       # (B, T_pfeat, D_enc)

        # (선택) 마스크
        mask = None

        # 어텐션 미리 계산
        enc_proj_h = self.attn.precompute(enc_h)
        enc_proj_p = self.attn.precompute(enc_p)

        # 디코딩
        dec_out_h = self._decode_loop(enc_h, labels_h, tf_ratio)  # (B, T_h-1, V)
        dec_out_p = self._decode_loop(enc_p, labels_p, tf_ratio)  # (B, T_p-1, V)

        return dec_out_h, dec_out_p

    @torch.no_grad()
    def inference(self, images_h=None, images_p=None, max_seq_len=100):
        """
        필요 도메인만 넘기면 그 도메인만 디코딩.
        반환: (outs_h, outs_p)  # 둘 중 하나는 None일 수 있음
        """
        outs_h, outs_p = None, None
        sos_id = CFG["sos_idx"]

        if images_h is not None:
            enc_h = self.encoder(images_h)
            enc_h = self.encoder_proj(enc_h)
            mask_h = None
            enc_proj_h = self.attn.precompute(enc_h)

            B = enc_h.size(0)
            prev_id     = torch.full((B,), self.sos_id, dtype=torch.long, device=enc_h.device)
            prev_token  = self.embed(prev_id)
            prev_hidden = torch.zeros((B, self.dec_hid_dim), device=enc_h.device)

            dec_out = []
            for _ in range(1, max_seq_len):
                context, _ = self.attn(enc_h, prev_hidden, mask=mask_h, enc_proj=enc_proj_h)
                logits_y, cur_hidden = self.decoder(prev_token, prev_hidden, context)
                dec_out.append(logits_y)

                next_ids    = logits_y.argmax(dim=-1)
                prev_token  = self.embed(next_ids)
                prev_hidden = cur_hidden

            outs_h = torch.stack(dec_out).transpose(0, 1)  # (B, max_seq_len-1, V)

        if images_p is not None:
            enc_p = self.encoder(images_p)
            enc_p = self.encoder_proj(enc_p)
            mask_p = None
            enc_proj_p = self.attn.precompute(enc_p)

            B = enc_p.size(0)
            prev_ids    = torch.full((B,), self.sos_id, dtype=torch.long, device=enc_p.device)
            prev_token  = self.embed(prev_ids)
            prev_hidden = torch.zeros((B, self.dec_hid_dim), device=enc_p.device)

            dec_out = []
            for _ in range(1, max_seq_len):
                context, _ = self.attn(enc_p, prev_hidden, mask=mask_p, enc_proj=enc_proj_p)
                logits_y, cur_hidden = self.decoder(prev_token, prev_hidden, context)
                dec_out.append(logits_y)

                next_ids    = logits_y.argmax(dim=-1)
                prev_token  = self.embed(next_ids)
                prev_hidden = cur_hidden

            outs_p = torch.stack(dec_out).transpose(0, 1)  # (B, max_seq_len-1, V)
        return outs_h, outs_p

## Loss

In [ ]:
import torch
import torch.nn as nn

class DualLoss(nn.Module):
    """
    L = CE_h + CE_p + [CE_up]
    - logits_* : (B, T-1, V)   # t=1..T-1 예측
    - tgt_*    : (B, T)        # [<sos> ... <eos> pad ...]
    """
    def __init__(self, ignore_index=0, match_weight=0.0):
        super().__init__()
        self.pad = ignore_index
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none')
        self.match_weight = float(match_weight)

    def _ce(self, logits, tgt):
        if logits is None or tgt is None:
            # 장치 맞춰서 0 스칼라 텐서 반환
            device = logits.device if logits is not None else (tgt.device if tgt is not None else 'cpu')
            return torch.zeros((), device=device)
        B, Tp1, V = logits.shape
        y = tgt[:, 1:]  # <sos> 제거 (eos 포함)
        assert y.shape == (B, Tp1), f"shape mismatch: logits={logits.shape}, y={y.shape}"
        per_tok = self.ce(logits.reshape(-1, V), y.reshape(-1))        # (B*(T-1),)
        valid  = (y != self.pad).reshape(-1).float()                   # (B*(T-1),)
        return (per_tok * valid).sum() / valid.sum().clamp_min(1)

    # 확실하게 맞는지 모름
    def _match(self, ctx_h, ctx_p, tgt_h=None, tgt_p=None, device='cpu'):
        if ctx_h is None or ctx_p is None:
            return torch.zeros((), device=device)
        per_tok = ((ctx_h - ctx_p) ** 2).mean(dim=-1)  # (B, T-1)

        if tgt_h is not None:
            mask = (tgt_h[:, 1:] != self.pad)
        else:
            mask = torch.ones_like(per_tok, dtype=torch.bool)
        if tgt_p is not None:
            mask = mask & (tgt_p[:, 1:] != self.pad)

        valid = mask.float()
        return (per_tok * valid).sum() / valid.sum().clamp_min(1)

    def forward(self,
                logits_h, tgt_h,
                logits_p, tgt_p,
                logits_up=None, tgt_up=None,
                ctx_h=None, ctx_p=None):
        ce_h  = self._ce(logits_h, tgt_h)
        ce_p  = self._ce(logits_p, tgt_p)
        ce_up = self._ce(logits_up, tgt_up) if (logits_up is not None and tgt_up is not None) else ce_h.new_tensor(0.)

        zero = ce_h.new_tensor(0.)
        l_match = self._match(ctx_h, ctx_p, tgt_h, tgt_p, device=ce_h.device) if self.match_weight != 0 else zero

        total = ce_h + ce_p + ce_up + self.match_weight * l_match
        return total, {"ce_h": ce_h.detach(), "ce_p": ce_p.detach(),
                       "ce_up": ce_up.detach(), "match": l_match.detach(), "total": total.detach()}

In [ ]:
# def _match(self, ctx_h, ctx_p, tgt_h=None, tgt_p=None):
#     dev = (ctx_h.device if ctx_h is not None else
#            (ctx_p.device if ctx_p is not None else torch.device('cpu')))
#     if ctx_h is None or ctx_p is None:
#         return torch.zeros((), device=dev)

#     # 시간축 길이 맞추기
#     L = min(ctx_h.size(1), ctx_p.size(1))
#     if tgt_h is not None: L = min(L, max(tgt_h.size(1) - 1, 0))
#     if tgt_p is not None: L = min(L, max(tgt_p.size(1) - 1, 0))

#     ctx_h = ctx_h[:, :L, :]
#     ctx_p = ctx_p[:, :L, :]
#     per_tok = ((ctx_h - ctx_p) ** 2).mean(dim=-1)  # (B, L)

#     # 마스크
#     if tgt_h is not None and tgt_p is not None:
#         mask = (tgt_h[:, 1:1+L] != self.pad) & (tgt_p[:, 1:1+L] != self.pad)
#     elif tgt_h is not None:
#         mask = (tgt_h[:, 1:1+L] != self.pad)
#     elif tgt_p is not None:
#         mask = (tgt_p[:, 1:1+L] != self.pad)
#     else:
#         mask = torch.ones_like(per_tok, dtype=torch.bool)

#     valid = mask.float()
#     return (per_tok * valid).sum() / valid.sum().clamp_min(1)

# Train

In [ ]:
# @torch.no_grad()
# def evaluate_epoch_greedy(model, loader, device, pad_id=0, sos_id=1, eos_id=2, max_len=256, max_batches=None):
#     model.eval()
#     k_list = (0,1,2,3)
#     sums = {k: 0.0 for k in k_list}
#     nb = 0

#     for bi, batch in enumerate(tqdm(loader, desc="Eval", leave=False)):
#         if (max_batches is not None) and (bi >= max_batches): break
#         img_hme = batch["img_hme"].to(device)   # 평가를 HME 기준으로 한다고 가정
#         tgt     = batch["formula"].to(device)

#         preds = model.decode_greedy(img_hme, sos_id=sos_id, eos_id=eos_id, max_len=max_len)  # (B, T_pred)
#         for k in k_list:
#             sums[k] += exprate_k(preds=preds, targets=tgt, k=k, pad_id=pad_id, sos_id=sos_id, eos_id=eos_id)
#         nb += 1

#     return {f"expr@{k}": (sums[k] / max(1, nb)) for k in k_list}

In [ ]:
# # (루프 밖, 1회만) criterion 없으면 생성
# if criterion is None:
#     criterion = DualLoss(
#         ignore_index=CFG["training"]["ignore_idx"],
#         match_weight=CFG["training"].get("match_weight", 0.0),
#     )

In [ ]:
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn

def train(model, train_loader, valid_loader=None, criterion=None, optimizer=None, scheduler=None, device="cuda"):
    EPOCHS = CFG["training"]["epochs"]
    GRAD_CLIP = CFG["training"]["grad_clip"]
    # pad_idx = CFG["training"]["ignore_idx"]
    eos_idx = CFG["eos_idx"]

    best_loss = float("inf")
    patience = 0
    log_dict = {"Train": [], "Valid": []}

    save_best = Path(CFG["paths"]["best_ckpt"])
    save_last = Path(CFG["paths"]["last_ckpt"])

    for epoch in range(1, EPOCHS+1):
        # train
        model.train()
        total_loss = 0.0
        total_expr_0, total_expr_1, total_expr_2, total_expr_3 = 0.0, 0.0, 0.0, 0.0
        num_batches = 0

        loop = tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{EPOCHS}", leave=True)

        for batch in loop:
            img_hme = batch["img_hme"].to(device)
            img_pme = batch["img_pme"].to(device)
            tgt = batch["formula"].to(device)

            pred_hme, pred_pme = model(img_hme, img_pme, labels)  # (B,T-1,V)
            labels = labels[:, 1:]                  # sos부분은 제외: (B,T) => (B,T-1)

            loss, loss_items = criterion(
                logits_h=pred_hme, tgt_h=tgt,   # HME
                logits_p=pred_pme, tgt_p=tgt,   # PME
                logits_up=None, tgt_up=None,    # (없으면 None)
                ctx_h=None, ctx_p=None          # (없으면 None)
            )
            pred_tokens = pred.argmax(dim=-1)       # (B,T-1,V) -> (B,T-1)

            optimizer.zero_grad()
            loss.backward()

            optimizer.zero_grad()
            loss.backward()

            if GRAD_CLIP and GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            with torch.no_grad():
                expr_0 = exprate_k(preds=pred_tokens, targets=labels, k=0, eos_idx=eos_idx)
                expr_1 = exprate_k(preds=pred_tokens, targets=labels, k=1, eos_idx=eos_idx)
                expr_2 = exprate_k(preds=pred_tokens, targets=labels, k=2, eos_idx=eos_idx)
                expr_3 = exprate_k(preds=pred_tokens, targets=labels, k=3, eos_idx=eos_idx)

            total_loss += float(loss.item())
            total_expr_0 += float(expr_0)
            total_expr_1 += float(expr_1)
            total_expr_2 += float(expr_2)
            total_expr_3 += float(expr_3)
            num_batches += 1

            loop.set_postfix(loss=loss.item(), expr0=expr_0, expr1=expr_1, expr2=expr_2, expr3=expr_3, lr=optimizer.param_groups[0]['lr'])

        avg_loss = total_loss / num_batches
        avg_expr_0 = total_expr_0 / num_batches
        avg_expr_1 = total_expr_1 / num_batches
        avg_expr_2 = total_expr_2 / num_batches
        avg_expr_3 = total_expr_3 / num_batches

        print(f"\n[Epoch {epoch}] Loss: {avg_loss:.4f}, "
              f"Expr@0: {expr_0:.4f}, Expr@1: {expr_1:.4f}, Expr@2: {expr_2:.4f}, Expr@3: {expr_3:.4f}")

        log_dict["train"].append({
            "epoch": epoch,
            "loss": avg_loss,
            "expr@0": expr_0,
            "expr@1": expr_1,
            "expr@2": expr_2,
            "expr@3": expr_3,
        })

        # 스케줄러 종류별 호출 방식 분기
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_loss)
            else:
                scheduler.step()

        torch.save({"model": model.state_dict()}, save_last)

        if epoch == 1 or avg_loss < best_loss:
            best_loss = avg_loss
            patience = 0
            save_best.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"model": model.state_dict()}, save_best)
            print("Best model saved.")
        else:
            patience += 1
            if patience >= CFG["training"]["early_stop_patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # validation
        model.eval()
        total_loss = 0.0
        total_expr_0, total_expr_1, total_expr_2, total_expr_3 = 0.0, 0.0, 0.0, 0.0
        num_batches = 0

        loop = tqdm(valid_loader, desc=f"[Valid] Epoch {epoch}/{EPOCHS}", leave=True)

        for batch in loop:

            imgs = batch["image"].to(device)
            labels = batch["formula"].to(device)

            pred = model(imgs, labels)              # (B,T-1,V)
            labels = labels[:, 1:]                  # sos 제외: (B,T) => (B,T-1)

            loss = criterion(pred.transpose(1,2), labels)  # nn.CrossEntropyLoss() input shape 규칙에 따라 (B,V,T-1)로 pred가 와야하므로 맞춰줌
            pred_tokens = pred.argmax(dim=-1)              # (B,T-1,V) -> (B,T-1)

            expr_0 = float(exprate_k(preds=pred_tokens, targets=labels, k=0, eos_idx=eos_idx))
            expr_1 = float(exprate_k(preds=pred_tokens, targets=labels, k=1, eos_idx=eos_idx))
            expr_2 = float(exprate_k(preds=pred_tokens, targets=labels, k=2, eos_idx=eos_idx))
            expr_3 = float(exprate_k(preds=pred_tokens, targets=labels, k=3, eos_idx=eos_idx))

            total_loss += float(loss.item())
            total_expr_0 += expr_0
            total_expr_1 += expr_1
            total_expr_2 += expr_2
            total_expr_3 += expr_3
            num_batches += 1

            loop.set_postfix(loss=loss.item(), expr0=expr_0, expr1=expr_1, expr2=expr_2, expr3=expr_3)

        avg_loss = total_loss / num_batches
        avg_expr_0 = total_expr_0 / num_batches
        avg_expr_1 = total_expr_1 / num_batches
        avg_expr_2 = total_expr_2 / num_batches
        avg_expr_3 = total_expr_3 / num_batches


        print(f"\n[Valid Epoch {epoch}] Loss: {avg_loss:.4f}, "
              f"Expr@0: {avg_expr_0:.4f}, Expr@1: {avg_expr_1:.4f}, Expr@2: {avg_expr_2:.4f}, Expr@3: {avg_expr_3:.4f}")

        log_dict["Valid"].append({
            "epoch": epoch,
            "loss": avg_loss,
            "expr@0": avg_expr_0,
            "expr@1": avg_expr_1,
            "expr@2": avg_expr_2,
            "expr@3": avg_expr_3,
        })
    return log_dict, best_loss

In [ ]:
# # CosineAnnealingWarmupRestart 커스텀 클래스

# import math
# from torch.optim.lr_scheduler import _LRScheduler

# class CosineAnnealingWarmupRestarts(_LRScheduler):

#     def __init__(self,
#                  optimizer: torch.optim.Optimizer,
#                  first_cycle_steps: int,
#                  cycle_mult: float = 1.,
#                  max_lr: float = 0.1,
#                  min_lr: float = 0.0001,
#                  warmup_steps: int = 0,
#                  gamma: float = 1.,
#                  last_epoch: int = -1):
#         assert warmup_steps < first_cycle_steps

#         self.first_cycle_steps = first_cycle_steps
#         self.cycle_mult = cycle_mult
#         self.base_max_lr = max_lr
#         # self.max_lr = max_lr
#         self.min_lr = min_lr
#         self.warmup_steps = warmup_steps
#         self.gamma = gamma

#         self.cur_cycle_steps = first_cycle_steps
#         self.cycle = 0
#         self.step_in_cycle = last_epoch

#         super(CosineAnnealingWarmupRestarts, self).__init__(optimizer, last_epoch)

#         self.init_lr()

#     def init_lr(self):
#         self.base_lrs = []
#         for param_group in self.optimizer.param_groups:
#             param_group['lr'] = self.min_lr
#             self.base_lrs.append(self.min_lr)

#     def get_lr(self):
#         if self.step_in_cycle == -1:
#             return self.base_lrs
#         elif self.step_in_cycle < self.warmup_steps:
#             return [(self.max_lr - base_lr) * self.step_in_cycle / self.warmup_steps + base_lr for base_lr in self.base_lrs]
#         else:
#             return [base_lr + (self.max_lr - base_lr) \
#                     * (1 + math.cos(math.pi * (self.step_in_cycle - self.warmup_steps) / (self.cur_cycle_steps - self.warmup_steps))) / 2
#                     for base_lr in self.base_lrs]

#     def step(self, epoch=None):
#         if epoch is None:
#             epoch = self.last_epoch + 1
#             self.step_in_cycle = self.step_in_cycle + 1
#             if self.step_in_cycle >= self.cur_cycle_steps:
#                 self.cycle += 1
#                 self.step_in_cycle = self.step_in_cycle - self.cur_cycle_steps
#                 self.cur_cycle_steps = int((self.cur_cycle_steps - self.warmup_steps) * self.cycle_mult) + self.warmup_steps

#         else:
#             if epoch >= self.first_cycle_steps:
#                 if self.cycle_mult == 1.:
#                     self.step_in_cycle = epoch % self.first_cycle_steps
#                     self.cycle = epoch // self.first_cycle_steps
#                 else:
#                     n = int(math.log((epoch / self.first_cycle_steps * (self.cycle_mult - 1) + 1), self.cycle_mult))
#                     self.cycle = n
#                     self.step_in_cycle = epoch - int(self.first_cycle_steps * (self.cycle_mult ** n - 1) / (self.cycle_mult - 1))
#                     self.cur_cycle_steps = self.first_cycle_steps * self.cycle_mult ** (n)

#             else:
#                 self.cur_cycle_steps = self.first_cycle_steps
#                 self.step_in_cycle = epoch

#         self.max_lr = self.base_max_lr * (self.gamma ** self.cycle)
#         self.last_epoch = math.floor(epoch)
#         for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
#             param_group['lr'] = lr

In [ ]:
import os, json, random
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# Vocab
vocab = Vocab()
vocab.load_from_txt(Path(CFG["data"]["vocab"]))
vocab_size = len(vocab)

# Model, Loss, Optimizer, Scheduler
model = DLAModel_Full(model_config=CFG["model"]).to(DEVICE)
criterion = DualLoss(ignore_index=CFG["training"]["ignore_idx"])
opt_name = CFG["training"].get("optimizer", "adadelta").lower()
params = model.parameters()

if opt_name == "adam":
    optimizer = torch.optim.Adam(params, lr=CFG["training"]["learning_rate"])
elif opt_name == "sgd":
    optimizer = torch.optim.SGD(params, lr=CFG["training"]["learning_rate"], momentum=0.9)
elif opt_name == "adadelta":
    optimizer = torch.optim.Adadelta(params, lr=CFG["training"]["learning_rate"])
else:
    raise ValueError(f"Unknown optimizer: {opt_name}")

#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
scheduler = CosineAnnealingWarmupRestarts(optimizer,
                 first_cycle_steps=200,
                 cycle_mult=1.,
                 max_lr=0.1,
                 min_lr=0.0001,
                 warmup_steps=0,
                 gamma=1.,
                 last_epoch=-1)

# Train loop
log_dict, loss_history, best_loss = train(model, trainloader, paired_loader=None, criterion=criterion,
                                          optimizer=optimizer, scheduler=scheduler, device=DEVICE)

Epoch 1/10:   1%|          | 7/884 [01:29<3:07:27, 12.82s/it, loss=9.65]


KeyboardInterrupt: 

In [ ]:
# # 결과 저장
# save_log(log_dict, CFG["paths"]["train_log_json"])

In [ ]:
# # 결과 시각화
# with open(CFG["paths"]["train_log_json"], "r", encoding="utf-8") as f:
#     loss_history = json.load(f)   # dict 형태로 로드됨
# plot_loss_curve(loss_history, metric="loss", overlap=False)

# Test

In [ ]:
# # 단일 도메인 이용하는 Lite 버전의 Test 함수 (두 도메인 모두 이용할 경우 customdataset 출력 딕셔너리 키 구성 다르므로 살짝 수정필요)

# def test(model, test_loader, criterion, device="cuda", log_interval=50):
#     model.eval()
#     pad_idx = CFG["training"]["ignore_idx"]

#     total_loss = 0.0
#     total_expr_0, total_expr_1, total_expr_2, total_expr_3 = 0.0, 0.0, 0.0, 0.0
#     num_batches = 0

#     with torch.no_grad():
#         loop = tqdm(test_loader, desc="Testing", leave=True)
#         for i, batch in enumerate(loop):
#             imgs = batch["image"].to(device)
#             labels = batch["formula"].to(device)

#             B = imgs.size(0)
#             # teacher forcing 없이 greedy decoding
#             shifted_labels = torch.cat(
#                 [labels[:, 1:], torch.full((B, 1), 2, device=labels.device)], dim=-1
#             )

#             pred, eos_mask = model(imgs, shifted_labels)

#             labels_clone = labels.clone()
#             labels_clone[eos_mask[:, :, 2]] = pad_idx

#             loss = criterion(
#                 pred.reshape(-1, CFG["model"]["vocab_size"]), labels_clone.reshape(-1)
#             )

#             pred_tokens = pred.argmax(dim=-1)
#             tgt_eval = shifted_labels

#             # metrics
#             expr_0 = exprate_k(pred_tokens, tgt_eval, k=0, ignore_idx=pad_idx)
#             expr_1 = exprate_k(pred_tokens, tgt_eval, k=1, ignore_idx=pad_idx)
#             expr_2 = exprate_k(pred_tokens, tgt_eval, k=2, ignore_idx=pad_idx)
#             expr_3 = exprate_k(pred_tokens, tgt_eval, k=3, ignore_idx=pad_idx)

#             total_loss += float(loss.item())
#             total_expr_0 += float(expr_0)
#             total_expr_1 += float(expr_1)
#             total_expr_2 += float(expr_2)
#             total_expr_3 += float(expr_3)
#             num_batches += 1

#             if i % log_interval == 0:
#                 loop.set_postfix(
#                     loss=loss.item(),
#                     expr0=expr_0,
#                     expr1=expr_1,
#                     expr2=expr_2,
#                     expr3=expr_3,
#                 )

#     avg_loss = total_loss / num_batches
#     avg_expr_0 = total_expr_0 / num_batches
#     avg_expr_1 = total_expr_1 / num_batches
#     avg_expr_2 = total_expr_2 / num_batches
#     avg_expr_3 = total_expr_3 / num_batches

#     print(
#         f"\n[Test] Loss: {avg_loss:.4f}, "
#         f"Expr@0: {avg_expr_0:.4f}, Expr@1: {avg_expr_1:.4f}, "
#         f"Expr@2: {avg_expr_2:.4f}, Expr@3: {avg_expr_3:.4f}"
#     )

#     return {
#         "loss": avg_loss,
#         "expr@0": avg_expr_0,
#         "expr@1": avg_expr_1,
#         "expr@2": avg_expr_2,
#         "expr@3": avg_expr_3,
#     }


# result_2014 = test(model, testloader_2014, criterion, device="cuda", log_interval=50)
# result_2016 = test(model, testloader_2016, criterion, device="cuda", log_interval=50)
# result_2019 = test(model, testloader_2019, criterion, device="cuda", log_interval=50)